In [ ]:
from sklearn.datasets import make_classification
import torch

In [ ]:
# Step 1: Create a synthetic classification dataset using sklearn
X, y = make_classification(
    n_samples=10,       # Number of samples
    n_features=2,       # Number of features
    n_informative=2,    # Number of informative features
    n_redundant=0,      # Number of redundant features
    n_classes=2,        # Number of classes
    random_state=42     # For reproducibility
)

In [ ]:
X

In [ ]:
X.shape

In [ ]:
y

In [ ]:
y.shape

In [ ]:
# Convert the data to PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [ ]:
X

In [ ]:
y

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [ ]:
dataset = CustomDataset(X, y)

In [ ]:
len(dataset)

In [ ]:
dataset[2]

In [ ]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=False)

In [ ]:
epochs=25

In [ ]:
# A tiny model matching our 2-feature input, so we can demonstrate a REAL
# training loop using the DataLoader batches above (not just printing them).
import torch.nn as nn

class TinyClassifier(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    return self.sigmoid(self.linear(x))

model = TinyClassifier(num_features=2)
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# NOTE: the original bug here was `for epoch in epochs:` -- `epochs` is just the
# int 25, not iterable, so that line crashes with "TypeError: 'int' object is
# not iterable". The fix is `range(epochs)`, exactly like every training loop
# elsewhere in this course.
for epoch in range(epochs):
  epoch_loss = 0.0
  for batch_features, batch_labels in dataloader:
    # forward pass
    preds = model(batch_features).squeeze(1)          # shape (batch,) to match labels
    loss = loss_fn(preds, batch_labels.float())        # BCELoss needs float targets

    # backward pass + update (the standard 5-step rhythm)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()

  if epoch % 5 == 0 or epoch == epochs - 1:
    print(f"epoch {epoch:2d}  avg loss {epoch_loss/len(dataloader):.4f}")
